# 🚰 Day 04a: tf.data — Efficient Data Pipelines

---

## 🎯 What You'll Master Today
- `tf.data.Dataset` — the TF data pipeline
- `from_tensor_slices`, `from_generator`
- `map`, `batch`, `shuffle`, `prefetch`
- Performance optimization patterns

**Goal:** Build production-grade data pipelines from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  tf.data Pipeline Anatomy                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Source → Transform → Batch → Prefetch → model.fit()           ║
║                                                                   ║
║  ds = tf.data.Dataset.from_tensor_slices((X, y))                ║
║  ds = ds.shuffle(buffer_size)                                    ║
║  ds = ds.map(preprocess_fn, num_parallel_calls=AUTOTUNE)        ║
║  ds = ds.batch(32)                                               ║
║  ds = ds.prefetch(AUTOTUNE)                                     ║
║                                                                   ║
║  Golden rule: shuffle → map → batch → prefetch                 ║
║                                                                   ║
║  PyTorch equivalent:                                             ║
║    Dataset + DataLoader → tf.data.Dataset                       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import tensorflow as tf
import numpy as np

AUTOTUNE = tf.data.AUTOTUNE

# ============================================================
# 1. Creating Datasets
# ============================================================

# From numpy arrays (most common)
X = np.random.randn(1000, 10).astype(np.float32)
y = np.random.randint(0, 3, 1000).astype(np.int32)

ds = tf.data.Dataset.from_tensor_slices((X, y))
print(f"Dataset: {ds}")
print(f"Element spec: {ds.element_spec}")

# Take a few samples
for features, label in ds.take(3):
    print(f"  Features shape: {features.shape}, Label: {label.numpy()}")

# From tensors only (no labels)
ds_x = tf.data.Dataset.from_tensor_slices(X)

# Range dataset
ds_range = tf.data.Dataset.range(10)
print(f"\nRange: {list(ds_range.as_numpy_iterator())}")

# From generator (for large/custom data)
def my_generator():
    for i in range(100):
        yield np.random.randn(10).astype(np.float32), np.int32(i % 3)

ds_gen = tf.data.Dataset.from_generator(
    my_generator,
    output_signature=(
        tf.TensorSpec(shape=(10,), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )
)
print(f"Generator dataset: {ds_gen}")

In [ ]:
# ============================================================
# 2. Core Transformations
# ============================================================

ds = tf.data.Dataset.from_tensor_slices((X, y))

# ----- shuffle -----
# buffer_size: how many elements to randomize at once
# Larger buffer = better shuffle, more memory
ds_shuffled = ds.shuffle(buffer_size=1000, seed=42)

# ----- map: apply a function per element -----
def preprocess(features, label):
    # Normalize features
    features = (features - tf.reduce_mean(features)) / (tf.math.reduce_std(features) + 1e-7)
    # One-hot encode label
    label_oh = tf.one_hot(label, depth=3)
    return features, label_oh

ds_mapped = ds_shuffled.map(preprocess, num_parallel_calls=AUTOTUNE)

# ----- batch -----
ds_batched = ds_mapped.batch(32)

# ----- prefetch: overlap data loading with training -----
ds_final = ds_batched.prefetch(AUTOTUNE)

# Check output
for batch_x, batch_y in ds_final.take(1):
    print(f"Batch X: {batch_x.shape}")
    print(f"Batch Y: {batch_y.shape}")
    print(f"Y[0]: {batch_y[0].numpy()} (one-hot)")

print("\n💡 Order: shuffle → map → batch → prefetch")
print("   AUTOTUNE lets TF decide parallelism dynamically")

In [ ]:
# ============================================================
# 3. Production Pipeline Template
# ============================================================

def build_dataset(X, y, batch_size=32, shuffle=True, augment_fn=None):
    """Production-ready tf.data pipeline."""
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    
    return ds

# Usage
train_ds = build_dataset(X[:800], y[:800], batch_size=32, shuffle=True)
val_ds = build_dataset(X[800:], y[800:], batch_size=32, shuffle=False)

# Train with tf.data
from tensorflow.keras import layers

model = tf.keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dense(3, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# fit() accepts tf.data.Dataset directly!
history = model.fit(train_ds, validation_data=val_ds, epochs=5, verbose=1)
print("\n💡 model.fit() works directly with tf.data.Dataset — no need for numpy")

In [ ]:
# ============================================================
# 4. Useful Operations
# ============================================================

ds = tf.data.Dataset.range(20)

# ----- repeat: loop dataset indefinitely -----
ds_repeat = ds.repeat(2)  # repeat 2 times
print(f"Repeat(2): {list(ds_repeat.as_numpy_iterator())}")

# ----- filter -----
ds_even = ds.filter(lambda x: x % 2 == 0)
print(f"Even only: {list(ds_even.as_numpy_iterator())}")

# ----- window (for time series) -----
ds_windows = ds.window(size=3, shift=1, drop_remainder=True)
ds_windows = ds_windows.flat_map(lambda w: w.batch(3))
print(f"Windows: {[w.numpy() for w in ds_windows.take(5)]}")

# ----- cache: cache in memory or on disk -----
ds_cached = tf.data.Dataset.from_tensor_slices(X).cache()  # in-memory
# ds_cached = ds.cache('/tmp/cache')  # on-disk for large datasets

# ----- zip: combine datasets -----
ds_x = tf.data.Dataset.from_tensor_slices([1, 2, 3])
ds_y = tf.data.Dataset.from_tensor_slices(['a', 'b', 'c'])
ds_zipped = tf.data.Dataset.zip((ds_x, ds_y))
print(f"Zipped: {list(ds_zipped.as_numpy_iterator())}")

# ----- cardinality -----
print(f"\nCardinality: {tf.data.experimental.cardinality(ds).numpy()}")

print("\n💡 cache() before shuffle/map → avoid re-reading data each epoch")
print("   window() → essential for time-series sliding windows")

In [ ]:
# ============================================================
# 5. Performance Best Practices
# ============================================================

print("""
╔═══════════════════════════════════════════════════════════════════╗
║  tf.data Performance Checklist                                   ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. prefetch(AUTOTUNE)         Always. Overlaps IO with compute ║
║  2. map(num_parallel_calls)    Parallelize preprocessing        ║
║  3. cache()                    If data fits in memory           ║
║  4. Interleave for disk IO     Read from multiple files         ║
║                                                                   ║
║  Optimal pipeline order:                                         ║
║    source → cache → shuffle → map → batch → prefetch            ║
║                                                                   ║
║  Common mistakes:                                                ║
║    ✗ shuffle after batch (shuffles batches, not elements)       ║
║    ✗ map before cache (re-processes every epoch)                ║
║    ✗ no prefetch (GPU idle while waiting for data)              ║
║    ✗ small shuffle buffer (poor randomization)                  ║
║                                                                   ║
║  tf.data vs PyTorch DataLoader:                                  ║
║    ┌──────────────┬───────────────────────────────┐             ║
║    │ PyTorch      │ tf.data                       │             ║
║    │ Dataset      │ from_tensor_slices / generator│             ║
║    │ DataLoader   │ .batch().prefetch()            │             ║
║    │ num_workers  │ num_parallel_calls=AUTOTUNE   │             ║
║    │ collate_fn   │ .map(fn)                      │             ║
║    └──────────────┴───────────────────────────────┘             ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is tf.data.Dataset? | Lazy pipeline for data loading. Supports map, batch, shuffle, prefetch. Integrated with model.fit() |
| 2 | Why prefetch? | Overlaps data loading (CPU) with model training (GPU). AUTOTUNE lets TF decide buffer size |
| 3 | Correct pipeline order? | source → cache → shuffle → map → batch → prefetch |
| 4 | shuffle buffer_size matters? | buffer_size=len(data) for perfect shuffle. Small buffer = poor randomization. Trade-off with memory |
| 5 | cache before or after map? | Before map if transform is expensive (avoids re-processing). After map if caching processed data fits |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • from_tensor_slices((X, y)) for numpy arrays          │
## │  • Order: cache → shuffle → map → batch → prefetch     │
## │  • Always use prefetch(AUTOTUNE)                         │
## │  • map(fn, num_parallel_calls=AUTOTUNE)                 │
## │  • model.fit() accepts Dataset directly                  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Transfer Learning** — pretrained models